In [2]:
%run C:/Users/caleb/ML2Project/AMMCalc.ipynb

In [5]:
import os
import pandas as pd
import numpy as np

# Goes in my external drive and sets directory
RAW_DIR = r"D:\MLHFTData\DecompressedData"
OUT_DIR = os.path.join(os.getcwd(), "features")
os.makedirs(OUT_DIR, exist_ok=True)

# All days in set, used for matching
DAYS = ["20251111", "20251112", "20251113", "20251114", "20251117", "20251118", "20251119", "20251120", "20251121", "20251124", 
        "20251125", "20251126", "20251128", "20251201", "20251202", "20251203", "20251204", "20251205", "20251208", "20251209", 
        "20251210", "20251211", "20251212", "20251215", "20251216", "20251217", "20251218", "20251219", "20251222", "20251223",
        "20251224", "20251226", "20251229", "20251230", "20251231", "20260102", "20260105", "20260106", "20260107", "20260108", 
        "20260109", "20260112", "20260113", "20260114", "20260115", "20260116", "20260120", "20260121", "20260122", "20260123", 
        "20260126", "20260127", "20260128", "20260129", "20260130", "20260202", "20260203", "20260204", "20260205", "20260206",
        "20260209"]

# Split data loading to not overwhelme machine
CHUNK_SIZE = 500000
DOWNSAMPLE = 100
DEPTH = 10


def loadFeaturesForDay(day: str, rawDir: str = RAW_DIR, chunkSize: int = CHUNK_SIZE, downsample: int = DOWNSAMPLE, depth: int = DEPTH, verbose: bool = True) -> pd.DataFrame:
    # File path
    fp = os.path.join(rawDir, f"xnas-itch-{day}.mbp-10.csv")
    # stop if doesn't exist
    if not os.path.exists(fp):
        raise FileNotFoundError(fp)

    # Set col names by value
    bidPxCols = [f"bid_px_{i:02d}" for i in range(depth)]
    askPxCols = [f"ask_px_{i:02d}" for i in range(depth)]
    bidSzCols = [f"bid_sz_{i:02d}" for i in range(depth)]
    askSzCols = [f"ask_sz_{i:02d}" for i in range(depth)]
    bidCtCols = [f"bid_ct_{i:02d}" for i in range(depth)]
    askCtCols = [f"ask_ct_{i:02d}" for i in range(depth)]
    # From original data
    baseCols = ["ts_recv", "action", "side"]
    usecols = baseCols + bidPxCols + askPxCols + bidSzCols + askSzCols + bidCtCols + askCtCols
    # Generate features from AMMCalc
    engine = FeatureEngine()
    feats = []
    totalRows = 0

    for chunkIdx, chunk in enumerate(pd.read_csv(fp, usecols=usecols, chunksize=chunkSize),start=1):
        # Index total rows
        totalRows += len(chunk)
        # take every downsample row
        if downsample > 1:
            chunk = chunk.iloc[::downsample].copy()

        chunk = chunk[(chunk["bid_px_00"] > 0) &(chunk["ask_px_00"] > 0)]
        # If empty, go on
        if chunk.empty:
            if verbose:
                print(f"Chunk {chunkIdx}: empty after filter")
            continue
        # Intake cols + the new features
        bidPx = chunk[bidPxCols].to_numpy(dtype=np.float32, copy=False)
        askPx = chunk[askPxCols].to_numpy(dtype=np.float32, copy=False)
        bidSz = chunk[bidSzCols].to_numpy(dtype=np.float32, copy=False)
        askSz = chunk[askSzCols].to_numpy(dtype=np.float32, copy=False)
        bidCt = chunk[bidCtCols].to_numpy(dtype=np.int32, copy=False)
        askCt = chunk[askCtCols].to_numpy(dtype=np.int32, copy=False)
        # Intake original features
        ts = chunk["ts_recv"].to_numpy(dtype=np.int64, copy=False)
        action = chunk["action"].astype(str).to_numpy()
        side = chunk["side"].astype(str).to_numpy()
        # Original Snapshot
        for i in range(len(ts)):
            snap = LOBSnapshot(timestamp = int(ts[i]), bid_prices = bidPx[i], ask_prices = askPx[i], bid_sizes = bidSz[i], ask_sizes = askSz[i],
                               bid_counts = bidCt[i], ask_counts = askCt[i], action = action[i] if action[i] else "N", side = side[i] if side[i] else "N")
            f = engine.createFeatures(snap)
            if f is not None:
                feats.append(f)
        if verbose:
            print(f"Chunk {chunkIdx}: kept={len(ts):,} | featsTotal={len(feats):,}")
        del chunk, bidPx, askPx, bidSz, askSz, bidCt, askCt, ts, action, side

    df = pd.DataFrame(feats)

    # Adapt to AMMEnvironment in AMMCalc
    if len(df) > 0:
        envCols = list(AMMEnvironment.COLS)
        missing = [c for c in envCols if c not in df.columns]
        if missing:
            raise ValueError(f"FeatureEngine output missing required columns: {missing}")

        keepCols = (["timestamp"] + envCols) if "timestamp" in df.columns else envCols
        df = df[keepCols].copy()

        numericCols = [c for c in df.columns if c != "timestamp"]
        arr = df[numericCols].to_numpy(dtype=np.float64, copy=False)
        df = df.loc[np.isfinite(arr).all(axis=1)].reset_index(drop=True)

    if verbose:
        print(f"Day {day} done. rawRows={totalRows:,} | features={len(df):,}")

    return df


def loadAllDaysAndSplit(days=DAYS, trainFrac=0.7, valFrac=0.15):
    allDayFeatures = []
    for i, d in enumerate(days):
        df = loadFeaturesForDay(d)
        allDayFeatures.append(df)
        print(f"Loaded {i+1}/{len(days)} days")
    
    n = len(allDayFeatures)
    trainEnd = int(n * trainFrac)
    valEnd   = int(n * (trainFrac + valFrac))
    dfTrain = pd.concat(allDayFeatures[:trainEnd], ignore_index=True)
    dfVal   = pd.concat(allDayFeatures[trainEnd:valEnd], ignore_index=True)
    dfTest  = pd.concat(allDayFeatures[valEnd:], ignore_index=True)
    return dfTrain, dfVal, dfTest, allDayFeatures


In [3]:
print(dfTrain.shape, dfVal.shape, dfTest.shape)
print(dfTrain.columns.tolist())

# basic finiteness check
numCols = [c for c in dfTrain.columns if c != "timestamp"]
print("Non-finite rows (train):", (~np.isfinite(dfTrain[numCols].to_numpy())).any(axis=1).sum())

# quick distribution glance
print(dfTrain[AMMEnvironment.COLS].describe().T[["mean","std","min","max"]].head(10))

(350235, 24) (161351, 24) (158605, 24)
['timestamp', 'mid_price', 'microprice', 'spread', 'spread_bps', 'imbalance_l1', 'bid_vwap_5', 'ask_vwap_5', 'bid_depth_5', 'ask_depth_5', 'depth_imbalance_5', 'bid_slope', 'ask_slope', 'vol_500tick', 'hawkes_lambda_market', 'hawkes_integrated_market', 'add_rate', 'cancel_rate', 'cancel_add_ratio', 'order_intensity', 'price_change', 'price_change_bps', 'best_bid', 'best_ask']
Non-finite rows (train): 0
                          mean          std         min           max
mid_price           678.542781     4.211409  670.515015    685.734985
microprice          678.542622     4.211221  670.516683    685.748799
spread                0.013302     0.015620    0.009949      0.900024
spread_bps            0.195964     0.229215    0.145217     13.195872
imbalance_l1         -0.011965     0.474032   -0.999716      0.999618
bid_vwap_5          678.510832     4.208563  670.488946    685.708002
ask_vwap_5          678.575396     4.214708  670.545027    686.24